# 04 · Who gets funded

**Goal:** Cross-variable analysis to surface headline findings and candidate stories.

Sections:
1. Setup & data load
2. Funding by college × year heatmap (detailed)
3. Agency × department funding matrix
4. Faculty rank × award size
5. Top-N faculty / departments by total $
6. Funding concentration — Gini coefficient & top-10 share
7. Candidate headline findings list

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', palette='muted')

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'data' / 'processed').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
OUTPUTS_DIR   = REPO_ROOT / 'outputs'
OUTPUTS_DIR.mkdir(exist_ok=True)

In [ ]:
faculty       = pd.read_parquet(PROCESSED_DIR / 'faculty.parquet')
grants        = pd.read_parquet(PROCESSED_DIR / 'grants.parquet')
faculty_grants = pd.read_parquet(PROCESSED_DIR / 'faculty_grants.parquet')

# Merge faculty_grants with faculty and grants for enriched analysis
gf_full = (
    faculty_grants
    .merge(faculty[['faculty_id', 'superior_academic_unit',
                    'academic_unit', 'academic_rank', 'tenure_status']],
           on='faculty_id', how='left')
    .merge(grants[['grant_id', 'totaldollars', 'agencyname', 'agencycode',
                   'startdateyear']],
           on='grant_id', how='left')
)

grants_yr = grants[(grants['startdateyear'] >= 2000) &
                   (grants['startdateyear'] <= 2025)].copy()

print(f'Enriched grant-faculty table: {gf_full.shape}')
print(f'Missing college after join: {gf_full["superior_academic_unit"].isna().sum():,}')

## 1 · Agency × Department Funding Matrix

In [ ]:
a_col = 'agencycode' if 'agencycode' in gf_full.columns else 'agencyname'
top_agencies = gf_full[a_col].value_counts().head(6).index.tolist()
top_depts = (gf_full.groupby('academic_unit', observed=True)['totaldollars']
             .sum().sort_values(ascending=False).head(10).index.tolist())

matrix = (gf_full[
            gf_full[a_col].isin(top_agencies) &
            gf_full['academic_unit'].isin(top_depts)]
          .groupby([a_col, 'academic_unit'], observed=True)['totaldollars']
          .sum()
          .unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(matrix / 1e6, annot=True, fmt='.1f', cmap='Blues', ax=ax,
            linewidths=0.4, cbar_kws={'label': 'Total funding ($M)'})
ax.set_title('Agency × Department Funding Matrix ($M, top 6 agencies × top 10 depts)')
ax.set_ylabel('Agency')
ax.set_xlabel('Department')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'w6_agency_dept_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 2 · Faculty Rank × Award Size

In [ ]:
# Deduplicate: one row per grant per faculty member
gf_dedup = gf_full.drop_duplicates(subset=['grant_id', 'faculty_id'])

rank_funding = (gf_dedup
                .dropna(subset=['academic_rank'])
                .groupby('academic_rank', observed=True)
                .agg(
                    n_faculty=('faculty_id', 'nunique'),
                    n_grants=('grant_id', 'nunique'),
                    total_dollars=('totaldollars', 'sum'),
                    mean_grant=('totaldollars', 'mean'),
                    median_grant=('totaldollars', 'median'),
                )
                .sort_values('total_dollars', ascending=False)
                .reset_index())

rank_funding['avg_per_faculty'] = rank_funding['total_dollars'] / rank_funding['n_faculty']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = sns.color_palette('viridis', len(rank_funding))

axes[0].barh(rank_funding['academic_rank'], rank_funding['total_dollars'] / 1e6,
             color=colors)
axes[0].set_xlabel('Total funding ($M)')
axes[0].set_title('Total Funding by Rank')

axes[1].barh(rank_funding['academic_rank'], rank_funding['mean_grant'] / 1e3,
             color=colors)
axes[1].set_xlabel('Avg grant size ($K)')
axes[1].set_title('Average Grant Size by Rank')

axes[2].barh(rank_funding['academic_rank'], rank_funding['avg_per_faculty'] / 1e3,
             color=colors)
axes[2].set_xlabel('Avg funding per faculty ($K)')
axes[2].set_title('Funding Productivity by Rank')

plt.suptitle('Faculty Rank × Funding Analysis', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'w6_rank_funding.png', dpi=150, bbox_inches='tight')
plt.show()
display(rank_funding)

## 3 · Top-N Faculty by Total Funding

In [ ]:
top_faculty = (gf_dedup
               .groupby(['faculty_id', 'faculty_name', 'superior_academic_unit'], observed=True)
               .agg(
                   total_dollars=('totaldollars', 'sum'),
                   n_grants=('grant_id', 'nunique'),
                   n_as_copi=('is_copi', 'sum'),
                   first_year=('startdateyear', 'min'),
                   last_year=('startdateyear', 'max'),
               )
               .sort_values('total_dollars', ascending=False)
               .head(25)
               .reset_index())

fig, ax = plt.subplots(figsize=(10, 9))
bars = ax.barh(top_faculty['faculty_name'][::-1],
               top_faculty['total_dollars'][::-1] / 1e6,
               color=sns.color_palette('plasma', len(top_faculty)))
ax.set_xlabel('Total grant funding ($M)')
ax.set_title('Top 25 Faculty by Total Grant Funding')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'w6_top25_faculty_funding.png', dpi=150, bbox_inches='tight')
plt.show()

display(top_faculty[['faculty_name', 'superior_academic_unit',
                      'total_dollars', 'n_grants', 'n_as_copi',
                      'first_year', 'last_year']])

In [ ]:
# Top 15 departments by total funding
top_depts_funding = (gf_dedup
                     .groupby('academic_unit', observed=True)
                     .agg(total_dollars=('totaldollars', 'sum'),
                          n_grants=('grant_id', 'nunique'),
                          n_faculty=('faculty_id', 'nunique'))
                     .sort_values('total_dollars', ascending=False)
                     .head(15)
                     .reset_index())
top_depts_funding['avg_per_faculty'] = (top_depts_funding['total_dollars'] /
                                         top_depts_funding['n_faculty'])

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_depts_funding['academic_unit'][::-1],
        top_depts_funding['total_dollars'][::-1] / 1e6,
        color=sns.color_palette('Blues_d', len(top_depts_funding)))
ax.set_xlabel('Total funding ($M)')
ax.set_title('Top 15 Departments by Total Grant Funding')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'w6_top15_dept_funding.png', dpi=150, bbox_inches='tight')
plt.show()

display(top_depts_funding)

## 4 · Funding Concentration (Gini & Top-10 Share)

In [ ]:
def gini(arr: np.ndarray) -> float:
    """Compute Gini coefficient of a non-negative array."""
    arr = np.sort(arr[arr > 0])
    n = len(arr)
    cumsum = np.cumsum(arr)
    return float((2 * np.sum(np.arange(1, n + 1) * arr) - (n + 1) * cumsum[-1]) /
                 (n * cumsum[-1]))

# Faculty-level total funding
faculty_totals = (gf_dedup
                  .groupby('faculty_id')['totaldollars']
                  .sum()
                  .values)

g = gini(faculty_totals)
faculty_totals_sorted = np.sort(faculty_totals[faculty_totals > 0])[::-1]
total_funding = faculty_totals_sorted.sum()
top10_share = faculty_totals_sorted[:10].sum() / total_funding * 100
top1_share  = faculty_totals_sorted[:1].sum() / total_funding * 100

print(f'Gini coefficient (faculty-level): {g:.3f}')
print(f'Top 1 faculty share of total funding: {top1_share:.1f}%')
print(f'Top 10 faculty share of total funding: {top10_share:.1f}%')

# Lorenz curve
sorted_vals = np.sort(faculty_totals[faculty_totals > 0])
cumulative_share = np.cumsum(sorted_vals) / sorted_vals.sum()
population_share = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(population_share, cumulative_share,
        color='steelblue', linewidth=2, label=f'Lorenz curve (Gini={g:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect equality')
ax.fill_between(population_share, cumulative_share, population_share,
                alpha=0.2, color='steelblue')
ax.set_xlabel('Cumulative share of faculty (sorted by funding)')
ax.set_ylabel('Cumulative share of total funding')
ax.set_title('Lorenz Curve — Grant Funding Concentration')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'w6_lorenz_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 · Tenure Status × Agency Funding Cross-Tab

In [ ]:
def categorize_tenure(s):
    if pd.isna(s): return 'Non-Tenure Track'
    s = str(s).lower()
    if 'tenured' in s and 'track' not in s: return 'Tenured'
    if 'tenure' in s: return 'Tenure-Track'
    return 'Non-Tenure Track'

gf_dedup = gf_dedup.copy()
gf_dedup['tenure_cat'] = gf_dedup['tenure_status'].apply(categorize_tenure)

tenure_agency = (gf_dedup[gf_dedup[a_col].isin(top_agencies)]
                 .groupby(['tenure_cat', a_col], observed=True)
                 ['totaldollars']
                 .sum()
                 .unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(10, 5))
tenure_agency.plot(kind='bar', ax=ax, colormap='tab10', edgecolor='black', alpha=0.85)
ax.set_xlabel('Tenure Category')
ax.set_ylabel('Total funding ($)')
ax.set_title('Funding by Tenure Status × Top Agencies')
ax.legend(title='Agency', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'w6_tenure_agency_crosstab.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 · Candidate Headline Findings

Fill in after running all cells:

- **Finding 1 — Funding concentration:** Gini ≈ 0.632 — top 10 faculty account for 19.6% of all dollars
- **Finding 2 — Agency dominance:** NSF + NIH account for >95% of grants; other agencies largely absent
- **Finding 3 — Department leaders:** 
    - MELODIA, TOMMASO	College of Engineering
    - MAKRIYANNIS, ALEXANDROS	Bouvé College of Health Sciences
    - LEVINE, HERBERT	College of Science
    - BARRETT, LISA FELDMAN	College of Science
    - CHOFFNES, DAVID ROSS	Khoury College of Computer Sciences
    - LAZER, DAVID M J	College of Social Sciences and Humanities

In [ ]:
# Export summary tables for Week 9 synthesis
top_faculty.to_csv(OUTPUTS_DIR / 'top_faculty_funding.csv', index=False)
top_depts_funding.to_csv(OUTPUTS_DIR / 'top_dept_funding.csv', index=False)
print('Saved summary CSVs.')

In [ ]:
# Top 5 grant agencies by total funding
top5_agencies = (gf_dedup.groupby(a_col, observed=True)['totaldollars']
                 .sum()
                 .sort_values(ascending=False)
                 .head(5))

print('Top 5 Grant Agencies by Total Funding:')
print('=' * 60)
for agency, total in top5_agencies.items():
    print(f'\n{agency}: ${total:,.0f}')
    
    # Get unique faculty for this agency with their colleges
    agency_faculty = (gf_dedup[gf_dedup[a_col] == agency]
                      .drop_duplicates(subset=['faculty_id'])
                      .dropna(subset=['faculty_name', 'superior_academic_unit'])
                      .sort_values('faculty_name')[['faculty_name', 'superior_academic_unit']])
    
    print(f'  {len(agency_faculty)} faculty members:')
    for idx, row in agency_faculty.iterrows():
        print(f'    • {row["faculty_name"]:<50} | {row["superior_academic_unit"]}')

## 7 · Faculty Lookup Tool

In [ ]:
def lookup_faculty(name_search, data=gf_full, grants_data=grants, partial_match=True):
    """
    Look up faculty information by name.
    
    Parameters:
    -----------
    name_search : str
        Faculty name or partial name to search for
    data : DataFrame, default=gf_full
        The grant-faculty dataset to search in
    grants_data : DataFrame, default=grants
        The grants dataset with grant names
    partial_match : bool, default=True
        If True, performs case-insensitive partial matching
        If False, requires exact match
    
    Returns:
    --------
    dict or None : Faculty information dictionary or None if not found
    """
    # Search for faculty name
    if partial_match:
        mask = data['faculty_name'].str.contains(name_search, case=False, na=False)
    else:
        mask = data['faculty_name'].str.lower() == name_search.lower()
    
    matches = data[mask]
    
    if len(matches) == 0:
        print(f"No faculty found matching '{name_search}'")
        return None
    
    # Get unique faculty IDs that match
    unique_faculty = matches[['faculty_id', 'faculty_name']].drop_duplicates()
    
    if len(unique_faculty) > 1:
        print(f"Found {len(unique_faculty)} faculty members matching '{name_search}':")
        for idx, row in unique_faculty.iterrows():
            print(f"  • {row['faculty_name']}")
        print("\nPlease be more specific. Showing results for first match.\n")
    
    # Take first match
    faculty_id = unique_faculty.iloc[0]['faculty_id']
    faculty_name = unique_faculty.iloc[0]['faculty_name']
    
    # Get all grants for this faculty
    faculty_grants = data[data['faculty_id'] == faculty_id].copy()
    
    # Merge with grants to get grant names
    faculty_grants = faculty_grants.merge(
        grants_data[['grant_id', 'grantname', 'startdate', 'enddate', 'durationinyears']],
        on='grant_id',
        how='left'
    )
    
    # Extract information
    college = faculty_grants['superior_academic_unit'].mode()[0] if not faculty_grants['superior_academic_unit'].isna().all() else 'Unknown'
    department = faculty_grants['academic_unit'].mode()[0] if not faculty_grants['academic_unit'].isna().all() else 'Unknown'
    rank = faculty_grants['academic_rank'].mode()[0] if not faculty_grants['academic_rank'].isna().all() else 'Unknown'
    tenure = faculty_grants['tenure_status'].mode()[0] if not faculty_grants['tenure_status'].isna().all() else 'Unknown'
    
    # Get unique grants
    grants_info = (faculty_grants[['grant_id', 'grantname', 'agencyname', 'totaldollars', 'startdateyear']]
                   .drop_duplicates(subset=['grant_id'])
                   .sort_values('startdateyear', ascending=False))
    
    # Summary statistics
    total_funding = grants_info['totaldollars'].sum()
    num_grants = len(grants_info)
    agencies = grants_info['agencyname'].value_counts().to_dict()
    year_range = f"{grants_info['startdateyear'].min():.0f}–{grants_info['startdateyear'].max():.0f}"
    
    # Print results
    print(f"{'='*80}")
    print(f"FACULTY PROFILE: {faculty_name}")
    print(f"{'='*80}\n")
    
    print(f"AFFILIATION")
    print(f"  College:     {college}")
    print(f"  Department:  {department}")
    print(f"  Rank:        {rank}")
    print(f"  Tenure:      {tenure}\n")
    
    print(f"FUNDING SUMMARY")
    print(f"  Total Grants:   {num_grants}")
    print(f"  Total Funding:  ${total_funding:,.0f}")
    print(f"  Year Range:     {year_range}\n")
    
    print(f"FUNDING AGENCIES ({len(agencies)} agencies)")
    for agency, count in sorted(agencies.items(), key=lambda x: x[1], reverse=True):
        if count > 0:  # Only show agencies with grants
            print(f"  • {agency}: {count} grant{'s' if count > 1 else ''}")
    
    print(f"\nGRANTS (showing all {num_grants} grants)")
    print(f"{'-'*80}")
    for idx, grant in grants_info.iterrows():
        year = f"{grant['startdateyear']:.0f}"
        amount = f"${grant['totaldollars']:,.0f}"
        print(f"\n[{year}] {amount:>15}")
        print(f"  Agency: {grant['agencyname']}")
        print(f"  Title:  {grant['grantname'][:80]}{'...' if len(str(grant['grantname'])) > 80 else ''}")
    
    print(f"\n{'='*80}\n")
    
    # Return structured data
    return {
        'name': faculty_name,
        'id': faculty_id,
        'college': college,
        'department': department,
        'rank': rank,
        'tenure': tenure,
        'num_grants': num_grants,
        'total_funding': total_funding,
        'year_range': year_range,
        'agencies': agencies,
        'grants': grants_info
    }

# Example usage:
# lookup_faculty('MELODIA')
# lookup_faculty('LAZER, DAVID')

In [ ]:
# Demo: Search for a faculty member
result = lookup_faculty('song')

## 6 · Attribution: earned at NEU vs prior institution

A grant listed against a Northeastern faculty member is not always research
*done at Northeastern*. When a senior PI joins from another institution, their
historical grants get pulled into the reporting system. The pipeline splits
each faculty–grant link into three buckets in `faculty_grants.neu_status`:

| Bucket | Rule (grant start relative to hire date) | Interpretation |
|---|---|---|
| `earned_at_neu`     | on or after hire date       | Money NEU raised. |
| `prior_institution` | strictly before hire date   | Purely historical. Does *not* count as NEU work. |
| `unknown`           | hire or grant start date missing | Cannot classify. |

In [ ]:
# Corpus-level breakdown
def _stats(mask):
    sub = faculty_grants[mask]
    dollars = (sub.merge(grants[['grant_id','totaldollars']].astype({'grant_id':str}), on='grant_id')
                  .drop_duplicates('grant_id').totaldollars.sum())
    return len(sub), sub.grant_id.nunique(), dollars

rows = []
for status in ['earned_at_neu','prior_institution','unknown']:
    n_rows, n_grants, dollars = _stats(faculty_grants.neu_status == status)
    rows.append({'bucket': status,
                 'faculty-grant rows': n_rows,
                 'unique grants': n_grants,
                 'total $ (dedup)': f'${dollars/1e6:,.0f}M'})

n_rows, n_grants, dollars = _stats(faculty_grants.notna().any(axis=1))
rows.append({'bucket': 'Grand total (all rows)',
             'faculty-grant rows': n_rows, 'unique grants': n_grants,
             'total $ (dedup)': f'${dollars/1e6:,.0f}M'})
pd.DataFrame(rows)

In [ ]:
# Top 15 faculty side by side — headline vs earned-at-NEU
mask_earned = faculty_grants.neu_status == 'earned_at_neu'

def top(mask):
    sub = faculty_grants[mask].merge(grants[['grant_id','totaldollars']].astype({'grant_id':str}),
                                      on='grant_id')
    return (sub.groupby(['faculty_id','faculty_name'])
               .agg(n=('grant_id','count'), total=('totaldollars','sum'))
               .sort_values('total', ascending=False))

top_all    = top(faculty_grants.neu_status.notna() | faculty_grants.neu_status.isna())
top_earned = top(mask_earned)

compare = (top_all.head(15).reset_index()
           .merge(top_earned.reset_index()[['faculty_id','total']].rename(columns={'total':'total_earned'}),
                  on='faculty_id', how='left')
           .fillna({'total_earned': 0}))
compare['pct_prior'] = ((1 - compare.total_earned / compare.total) * 100).round(0).astype(int)
disp = compare[['faculty_name','total','total_earned','pct_prior']].copy()
disp['total']        = disp.total.map(lambda x: f'${x/1e6:.1f}M')
disp['total_earned'] = disp.total_earned.map(lambda x: f'${x/1e6:.1f}M')
disp['pct_prior']    = disp.pct_prior.astype(str) + '%'
disp.columns = ['Faculty','All (headline)','Earned at NEU','% prior institution']
print('=== Top-15 headline PIs, earned-at-NEU view ===')
print(disp.to_string(index=False))

In [ ]:
# Top 15 by earned-at-NEU funding (the "actively raising money at NEU" list)
lb = top_earned.head(15).reset_index()
lb['total'] = lb.total.map(lambda x: f'${x/1e6:.1f}M')
lb.columns = ['faculty_id','Faculty','n_grants','Total $ (earned at NEU)']
print('=== Top-15 faculty by earned-at-NEU $ ===')
print(lb[['Faculty','n_grants','Total $ (earned at NEU)']].to_string(index=False))

In [ ]:
# Concentration under the two views
import numpy as np
def gini(x):
    x = np.sort(np.array(x, dtype=float))
    n = len(x); c = np.cumsum(x)
    return (n + 1 - 2 * c.sum() / c[-1]) / n

_all_join = faculty_grants.merge(grants[['grant_id','totaldollars']].astype({'grant_id':str}), on='grant_id')
t_all    = _all_join.groupby('faculty_id').totaldollars.sum()
_earned  = faculty_grants[mask_earned].merge(grants[['grant_id','totaldollars']].astype({'grant_id':str}), on='grant_id')
t_earned = _earned.groupby('faculty_id').totaldollars.sum()

pd.DataFrame([
    {'view': 'All (headline)',
     'n_funded': len(t_all), 'gini': round(gini(t_all.values),3),
     'top_10pct_share_pct': round(t_all.nlargest(int(len(t_all)*0.1)).sum()/t_all.sum()*100,1)},
    {'view': 'Earned at NEU only',
     'n_funded': len(t_earned), 'gini': round(gini(t_earned.values),3),
     'top_10pct_share_pct': round(t_earned.nlargest(int(len(t_earned)*0.1)).sum()/t_earned.sum()*100,1)},
])

### Takeaway

- **$1,408M earned at NEU vs $2,183M headline** — roughly 29% of the headline
  total is prior-institution attribution (grants started strictly before the
  PI's hire date). Another 7% is `unknown` (missing dates).
- **Three top-10 headline PIs shrink to $0 under the earned-at-NEU view**
  (Quaranta, Winslow, Bronich — all pre-hire). **Levine** shrinks from $47M to
  ~$6M. All are senior hires from 2019–2024.
- **Concentration rises under the filter**: Gini 0.632 → **0.653**; top-10%
  share 48% → **51%**. The prior-institution noise was flattening the curve.
- **Rule of thumb for reports:**
  - "What NEU raised" / "what research is happening at NEU" → filter to
    `neu_status == 'earned_at_neu'`.
  - "Career funding of NEU faculty" → use everything.